<a href="https://colab.research.google.com/github/jefersonferreirafagundes-eng/MVP-ML-Analytics/blob/main/MVP_Engenharia_Dados_Desempenho_Alunos_EXECUTADO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## MVP — Engenharia de Dados
#Nome: Jeferson Ferreira Fagundes

Matrícula: 4052026000082

Data: 09/2026

Dataset: Student Productivity & Distraction Dataset, extraído da Kaggle.

Fonte: https://raw.githubusercontent.com/jefersonferreirafagundes-eng/MVP-ML-Analytics/refs/heads/main/student_productivity_distraction_dataset_20000.csv

**Pergunta central:** quais variáveis disponíveis apresentam maior associação com a nota final dos alunos?



In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_PATH = Path(r"/mnt/data/student_productivity_distraction_dataset_20000.csv")
OUTPUT_DIR = Path(r"/mnt/data/MVP_Engenharia_Dados_Desempenho_Alunos_EXECUTADO")
OUTPUT_DIR.mkdir(exist_ok=True)

df_raw = pd.read_csv(DATA_PATH)
print("Dimensões do dataset:", df_raw.shape)
print("Colunas:", df_raw.columns.tolist())
df_raw.head()

FileNotFoundError: [Errno 2] No such file or directory: '/mnt/data/MVP_Engenharia_Dados_Desempenho_Alunos_EXECUTADO'

## 1. Camada Bronze

A Bronze preserva os dados originais. Nesta execução local, ela é persistida em CSV.

In [ ]:
bronze_path = OUTPUT_DIR / "bronze_student_raw.csv"
df_bronze = df_raw.copy()
df_bronze.to_csv(bronze_path, index=False)
print("Bronze:", bronze_path)
print("Linhas:", len(df_bronze), "| Colunas:", len(df_bronze.columns))

## 2. Qualidade inicial

In [ ]:
quality = pd.DataFrame({
    "tipo": df_bronze.dtypes.astype(str),
    "nulos": df_bronze.isna().sum(),
    "unicos": df_bronze.nunique(dropna=False),
})
print("Duplicidades exatas:", df_bronze.duplicated().sum())
quality

In [ ]:
df_bronze.describe(include="all").T

## 3. Camada Silver

Tratamentos:
- remoção de duplicidades exatas;
- padronização de `gender`;
- validação de faixas físicas de horas/dia;
- preservação de todas as variáveis originais.

In [ ]:
df_silver = df_bronze.drop_duplicates().copy()
df_silver["gender"] = df_silver["gender"].astype(str).str.strip().str.title()

hour_cols = [
    "study_hours_per_day", "sleep_hours", "phone_usage_hours",
    "social_media_hours", "youtube_hours", "gaming_hours"
]
for c in hour_cols:
    df_silver.loc[~df_silver[c].between(0, 24), c] = np.nan

df_silver.loc[df_silver["age"] < 0, "age"] = np.nan
df_silver.loc[df_silver["final_grade"] < 0, "final_grade"] = np.nan

silver_path = OUTPUT_DIR / "silver_student_clean.csv"
df_silver.to_csv(silver_path, index=False)

print("Linhas Silver:", len(df_silver))
print("Nulos após validação:", int(df_silver.isna().sum().sum()))

## 4. Camada Gold e Feature Engineering

As variáveis derivadas abaixo são proxies. Tempos digitais podem se sobrepor.

In [ ]:
df_gold = df_silver.copy()
df_gold["total_screen_time_proxy"] = (
    df_gold["phone_usage_hours"] +
    df_gold["youtube_hours"] +
    df_gold["gaming_hours"]
)
df_gold["entertainment_hours"] = (
    df_gold["social_media_hours"] +
    df_gold["youtube_hours"] +
    df_gold["gaming_hours"]
)
df_gold["study_sleep_ratio"] = np.where(
    df_gold["sleep_hours"] > 0,
    df_gold["study_hours_per_day"] / df_gold["sleep_hours"],
    np.nan
)
df_gold["digital_entertainment_ratio"] = np.where(
    df_gold["study_hours_per_day"] > 0,
    df_gold["entertainment_hours"] / df_gold["study_hours_per_day"],
    np.nan
)

gold_path = OUTPUT_DIR / "gold_student_analysis.csv"
df_gold.to_csv(gold_path, index=False)
print("Gold:", gold_path)
df_gold.head()

## 5. Catálogo de dados

In [ ]:
catalog = pd.DataFrame([
    ["student_id","int","Identificador do registro","identificador","não usar como preditor"],
    ["age","numérico","Idade do estudante",">=0","explicativa"],
    ["gender","categórico","Gênero informado","categorias da fonte","explicativa"],
    ["study_hours_per_day","numérico","Horas de estudo/dia","0–24","explicativa"],
    ["sleep_hours","numérico","Horas de sono/dia","0–24","explicativa"],
    ["phone_usage_hours","numérico","Uso do celular/dia","0–24","explicativa"],
    ["social_media_hours","numérico","Uso de redes sociais/dia","0–24","explicativa"],
    ["youtube_hours","numérico","Uso do YouTube/dia","0–24","explicativa"],
    ["gaming_hours","numérico","Jogos/dia","0–24","explicativa"],
    ["breaks_per_day","numérico","Pausas por dia","conforme fonte","explicativa"],
    ["coffee_intake_mg","numérico","Consumo de cafeína","conforme fonte","explicativa"],
    ["exercise_minutes","numérico","Minutos de exercício","conforme fonte","explicativa"],
    ["assignments_completed","numérico","Atividades concluídas","conforme fonte","explicativa"],
    ["attendance_percentage","numérico","Percentual de frequência","0–100","explicativa"],
    ["stress_level","numérico","Nível de estresse","conforme fonte","explicativa"],
    ["focus_score","numérico","Índice de foco","conforme fonte","explicativa"],
    ["final_grade","numérico","Nota final","observado 40–99,99","alvo"],
    ["productivity_score","numérico","Índice de produtividade","derivado","avaliar antes de modelar"],
], columns=["atributo","tipo","descricao","dominio","papel"])
catalog.to_csv(OUTPUT_DIR / "catalogo_dados_executado.csv", index=False)
catalog

## 6. Distribuição da nota final

In [ ]:
print(df_gold["final_grade"].describe())

plt.figure(figsize=(8,4))
plt.hist(df_gold["final_grade"], bins=30)
plt.xlabel("Nota final")
plt.ylabel("Frequência")
plt.title("Distribuição de final_grade")
plt.show()

## 7. Correlação com a nota final

In [ ]:
numeric_cols = [
    c for c in df_gold.select_dtypes(include=np.number).columns
    if c not in ["student_id", "final_grade"]
]
corr = pd.DataFrame({
    "pearson": df_gold[numeric_cols + ["final_grade"]].corr(method="pearson")["final_grade"].drop("final_grade"),
    "spearman": df_gold[numeric_cols + ["final_grade"]].corr(method="spearman")["final_grade"].drop("final_grade"),
})
corr["abs_pearson"] = corr["pearson"].abs()
corr = corr.sort_values("abs_pearson", ascending=False)
corr.to_csv(OUTPUT_DIR / "correlacoes_final_grade.csv")
corr

In [ ]:
plot_corr = corr.sort_values("pearson")
plt.figure(figsize=(9,7))
plt.barh(plot_corr.index, plot_corr["pearson"])
plt.axvline(0, linewidth=1)
plt.xlabel("Correlação de Pearson com final_grade")
plt.title("Associação linear individual com a nota final")
plt.show()

## 8. Verificação de `productivity_score`

Antes da regressão multivariada, verificamos se `productivity_score` é independente das demais variáveis ou se é uma variável derivada.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

components = [
    "study_hours_per_day", "sleep_hours", "phone_usage_hours",
    "attendance_percentage", "stress_level", "focus_score"
]
aux = LinearRegression().fit(df_gold[components], df_gold["productivity_score"])
reconstructed = aux.predict(df_gold[components])
r2_productivity = r2_score(df_gold["productivity_score"], reconstructed)

print("R² de reconstrução de productivity_score:", r2_productivity)
pd.DataFrame({
    "variavel": components,
    "coeficiente": aux.coef_
})

### Decisão de modelagem

Como `productivity_score` é praticamente reconstruído pelas variáveis componentes, ele será excluído da análise explicativa principal para evitar multicolinearidade severa e dupla contagem da mesma informação.

## 9. Regressão múltipla sem `productivity_score`

In [ ]:
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm

base_features = [
    c for c in df_gold.columns
    if c not in [
        "student_id", "final_grade", "productivity_score",
        "total_screen_time_proxy", "entertainment_hours",
        "study_sleep_ratio", "digital_entertainment_ratio"
    ]
]

num_features = [c for c in base_features if df_gold[c].dtype != "object"]
cat_features = [c for c in base_features if df_gold[c].dtype == "object"]

X_num = pd.DataFrame(
    StandardScaler().fit_transform(df_gold[num_features]),
    columns=num_features, index=df_gold.index
)
X_cat = pd.get_dummies(df_gold[cat_features], drop_first=True, dtype=float)
X_ols = pd.concat([X_num, X_cat], axis=1)
X_ols = sm.add_constant(X_ols).astype(float)
y = df_gold["final_grade"].astype(float)

ols = sm.OLS(y, X_ols).fit()
print(ols.summary())

In [ ]:
ols_table = pd.DataFrame({
    "coeficiente": ols.params,
    "p_valor": ols.pvalues,
    "ic_min": ols.conf_int()[0],
    "ic_max": ols.conf_int()[1],
})
ols_table["abs_coef"] = ols_table["coeficiente"].abs()
ols_table = ols_table.sort_values("abs_coef", ascending=False)
ols_table.to_csv(OUTPUT_DIR / "regressao_ols.csv")
ols_table

## 10. Machine Learning — comparação com baseline

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import time

features_ml = base_features
X = df_gold[features_ml].copy()
y = df_gold["final_grade"].copy()

num = [c for c in features_ml if X[c].dtype != "object"]
cat = [c for c in features_ml if X[c].dtype == "object"]

prep = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), num),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), cat)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

models = {
    "DummyMean": DummyRegressor(strategy="mean"),
    "LinearRegression": Pipeline([("prep", prep), ("model", LinearRegression())]),
    "Ridge": Pipeline([("prep", prep), ("model", Ridge(alpha=1.0))]),
    "RandomForest": Pipeline([("prep", prep), ("model", RandomForestRegressor(
        n_estimators=300, random_state=42, n_jobs=-1
    ))]),
    "GradientBoosting": Pipeline([("prep", prep), ("model", GradientBoostingRegressor(random_state=42))]),
}

rows = []
fitted = {}
for name, model in models.items():
    start = time.perf_counter()
    if name == "DummyMean":
        model.fit(np.zeros((len(y_train),1)), y_train)
        pred = model.predict(np.zeros((len(y_test),1)))
    else:
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
    rows.append({
        "modelo": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": mean_squared_error(y_test, pred) ** 0.5,
        "R2": r2_score(y_test, pred),
        "tempo_s": time.perf_counter() - start
    })
    fitted[name] = model

results_df = pd.DataFrame(rows).sort_values("RMSE")
results_df.to_csv(OUTPUT_DIR / "metricas_modelos.csv", index=False)
results_df

In [ ]:
plt.figure(figsize=(8,4))
plot_results = results_df.sort_values("R2")
plt.barh(plot_results["modelo"], plot_results["R2"])
plt.axvline(0, linewidth=1)
plt.xlabel("R² no teste")
plt.title("Modelos versus baseline")
plt.show()

## 11. Permutation Importance — Ridge

Como nenhum modelo demonstra boa capacidade preditiva, a importância deve ser lida com cautela. Valores próximos de zero indicam pouca contribuição incremental.

In [ ]:
from sklearn.inspection import permutation_importance

ridge = fitted["Ridge"]
perm = permutation_importance(
    ridge, X_test, y_test,
    scoring="neg_root_mean_squared_error",
    n_repeats=30,
    random_state=42,
    n_jobs=-1
)
perm_df = pd.DataFrame({
    "variavel": X_test.columns,
    "importancia_media": perm.importances_mean,
    "desvio": perm.importances_std
}).sort_values("importancia_media", ascending=False)
perm_df.to_csv(OUTPUT_DIR / "permutation_importance.csv", index=False)
perm_df

## 12. Resposta objetiva ao problema

A execução deve ser interpretada com base em três evidências:

1. magnitude das correlações com `final_grade`;
2. regressão múltipla;
3. capacidade dos modelos de superar o `DummyRegressor`.

Se as correlações forem muito pequenas e os modelos apresentarem R² próximo de zero ou negativo, a conclusão é que **as variáveis disponíveis não explicam adequadamente a variação da nota final neste dataset**.

In [ ]:
top = corr.iloc[0]
best = results_df.iloc[0]
dummy = results_df[results_df["modelo"]=="DummyMean"].iloc[0]

print(f"Maior correlação absoluta: {corr.index[0]} = {top['pearson']:.4f}")
print(f"Melhor RMSE observado: {best['modelo']} = {best['RMSE']:.4f}")
print(f"RMSE DummyMean: {dummy['RMSE']:.4f}")
print(f"R² do melhor modelo por RMSE: {best['R2']:.4f}")

if corr["abs_pearson"].max() < 0.10 and results_df[results_df["modelo"]!="DummyMean"]["R2"].max() <= 0.05:
    print("\nCONCLUSÃO:")
    print("As associações individuais são muito fracas e os modelos não demonstram capacidade preditiva relevante sobre a nota final.")
    print("O dataset não oferece evidência suficiente para afirmar que as variáveis analisadas expliquem diferenças de final_grade.")

## 13. Limitações e próximos passos

- O dataset contém 5.999 registros, apesar do nome do arquivo sugerir 20.000.
- `productivity_score` é praticamente uma combinação de outras variáveis e deve ser tratado como variável derivada.
- O conjunto de dados não demonstra relações fortes com `final_grade`.
- Não é possível concluir causalidade.
- Para investigar fatores que realmente influenciam desempenho, um trabalho futuro deveria incorporar variáveis como histórico escolar, dificuldade das avaliações, qualidade/instituição, contexto socioeconômico, frequência longitudinal e intervenções pedagógicas.